# Advanced Retrieval-Augmented Generation (RAG) with Fusion Techniques

Welcome to the advanced section of our course, where we move beyond basic question-answering systems and tackle the complexities of real-world enterprise knowledge retrieval. This notebook demonstrates a sophisticated RAG pipeline incorporating **Retrieval-Augmented Generation (RAG)** principles, specifically utilizing an advanced technique called **RAG Fusion**.

In simple RAG, a single query is used to retrieve documents. However, complex questions often require multiple perspectives or sub-queries to fully understand the context. RAG Fusion addresses this limitation by having the Large Language Model (LLM) generate several related sub-queries for a single user prompt. It then executes retrieval for *each* sub-query and intelligently fuses the results using techniques like Reciprocal Rank Fusion (RRF). This dramatically increases the recall and relevance of the retrieved context, ensuring that even nuanced or multi-faceted questions yield comprehensive answers.

Mastering RAG Fusion is critical for building production-grade AI applications because it mitigates the "single query failure" problem—where a simple keyword search fails to capture the full semantic meaning of a complex question. Furthermore, by chaining this advanced retrieval step into a final generation chain (using `ChatPromptTemplate` and LangChain Expression Language), we learn how to orchestrate multiple components: embedding models, vector stores, multi-query logic, and generative LLMs. These patterns are foundational building blocks for more complex state machines and agents built with frameworks like LangGraph.

### Learning Objectives

Upon completing this notebook, you will be able to:

*   **Implement Advanced Chunking:** Utilize `RecursiveCharacterTextSplitter` to segment large documents while maintaining semantic coherence.
*   **Build Vector Stores:** Create and manage a persistent vector store using ChromaDB with OpenAI embeddings for efficient similarity search.
*   **Apply RAG Fusion:** Understand and implement the concept of multi-query generation, allowing an LLM to break down a complex query into multiple sub-queries for enhanced retrieval.
*   **Orchestrate Retrieval:** Use the `RAGFusion` component to execute parallel retrievals and fuse results, significantly improving context relevance over standard single-query methods.
*   **Construct Generation Chains:** Build robust LangChain pipelines by combining custom prompts (`ChatPromptTemplate`) with LLMs to ensure that generation is strictly grounded in the provided context.


### RAG Fusion Pipeline

This notebook demonstrates a complete RAG (Retrieval-Augmented Generation) pipeline built on top of our custom `RAGFusion` class.
The pipeline uses **LLM-generated sub-queries** to improve retrieval quality, then fuses the results using **Reciprocal Rank Fusion (RRF)**.

**Steps covered:**
1. Load the source PDF
2. Split documents into chunks
3. Generate embeddings and store in ChromaDB
4. Create a similarity-search retriever
5. Apply RAG Fusion (sub-query generation + RRF)
6. Augmentation - build context from retrieved documents
7. Generation - produce a grounded answer using an LLM

### Imports & Setup

### Setup and Imports

This cell initializes the environment by loading API keys from a `.env` file. It then imports all necessary components: document loaders (`PyPDFLoader`), text splitters, embedding models (`OpenAIEmbeddings`), chat models (`ChatOpenAI`), vector store (`Chroma`), prompt templates, and the custom `RAGFusion` class.


In [3]:
from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate

from rag_fusion import RAGFusion

# Load OPENAI_API_KEY from the .env file
load_dotenv()

True

### Step 1 - Load the PDF

`PyPDFLoader` reads the PDF and returns one `Document` object per page.

### Documenting Data Loading

This cell initializes a document loader using `PyPDFLoader` to read content from the specified PDF file (`notebooklm_rag.pdf`). The loaded documents are stored in the `pages` variable, allowing us to count and confirm that the data has been successfully ingested into the RAG pipeline.


In [4]:
loader = PyPDFLoader("notebooklm_rag.pdf")  # Initialize the loader with the PDF file path
pages = loader.load()  # Load all pages/documents from the specified PDF

print(f"Loaded {len(pages)} page(s) from the PDF.") # Print a confirmation message showing the number of loaded pages


Loaded 3 page(s) from the PDF.


### Step 2 - Split Documents into Chunks

Large pages are split into smaller, overlapping chunks so that the retriever can surface focused, relevant passages rather than entire pages.

### Document Chunking (Text Splitting)

This cell uses `RecursiveCharacterTextSplitter` to break down large documents (`pages`) into smaller, manageable chunks. This process is crucial for RAG because embedding models have token limits and retrieving small, focused pieces of text improves the relevance and accuracy of the retrieved context.


In [5]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

# Split the input documents (pages) into smaller chunks using the defined splitter.
chunks = splitter.split_documents(pages)

# Print the total number of resulting chunks to confirm successful splitting.
print(f"Split into {len(chunks)} chunk(s).")


Split into 19 chunk(s).


### Step 3 - Embeddings & Vector Store

Each chunk is converted into a dense vector using OpenAI's `text-embedding-3-small` model and stored in a ChromaDB vector store.
This makes semantic similarity search possible at query time.

### Vector Store Initialization

This cell initializes and populates a Chroma vector store. It uses the specified `OpenAIEmbeddings` model to convert text chunks into numerical embeddings, which are then stored in the local Chroma database collection named `notebooklm_rag`. This step makes the document content searchable by semantic similarity.


In [6]:
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small") # Initialize the embedding model using OpenAI's text-embedding-3-small

vectorstore = Chroma.from_documents(
    documents=chunks, # The list of pre-processed document chunks to be stored
    embedding=embedding_model, # The embedding function used for conversion
    collection_name="notebooklm_rag" # Specifies the name of the collection in Chroma
)

print("Vector store created successfully.") # Confirmation message upon successful creation and population.


Vector store created successfully.


### Step 4 - Create the Retriever

We configure a similarity-search retriever with `k=3`, meaning it will return the 3 most relevant chunks for any given query.

### Retriever Initialization

This step converts the underlying vector store into a specialized retriever object. The `as_retriever()` method makes it easy to query relevant documents by performing similarity searches, specifically requesting the top 3 most similar chunks (`k=3`).


In [7]:
retriever = vectorstore.as_retriever(
    search_type="similarity",  # Specifies that we are using cosine/Euclidean distance for similarity search
    search_kwargs={"k": 3}   # Limits the retrieval to the top 3 most relevant documents (chunks)
)


### Step 5 - RAG Fusion

`RAGFusion.from_llm` wires up the LLM to generate multiple sub-queries from the original query.
Each sub-query is sent to the retriever independently, and the results are merged using **Reciprocal Rank Fusion (RRF)** - documents that rank highly across multiple sub-queries bubble to the top.

### RAG Fusion Pipeline Initialization

This cell initializes the `RAGFusion` pipeline, which is an advanced retrieval technique. Instead of a single query, it uses the LLM to generate multiple sub-queries (here, 2) and retrieves documents for each one, finally fusing the results into a comprehensive answer.


In [10]:
llm = ChatOpenAI(model="gpt-5-mini")

# Build the RAG Fusion pipeline: LLM generates 2 sub-queries, retrieves docs for each, then fuses
rag_fusion = RAGFusion.from_llm(
    llm=llm,
    retriever=retriever,
    num_subqueries=2,  # Specifies that the LLM should generate 2 distinct sub-queries
    k=3                # Sets the number of top documents to retrieve for each sub-query
)


### Code Explanation

This cell executes the core Retrieval-Augmented Generation (RAG) fusion step. It takes a user query, uses `rag_fusion` to automatically generate multiple sub-queries and retrieve documents for each, and then combines these results using Reciprocal Rank Fusion (RRF) to provide a comprehensive set of relevant documents.


In [11]:
query = "How does NotebookLM retrieve relevant information from uploaded documents?"

# This generates sub-queries, retrieves docs for each, and returns RRF-ranked results
fused_docs = rag_fusion.invoke(query)

print(f"Retrieved {len(fused_docs)} fused document(s).")
for i, doc in enumerate(fused_docs):
    print(f"\n--- Document {i + 1} ---")
    print(doc.page_content)


Retrieved 3 fused document(s).

--- Document 1 ---
rather than simple keyword matching. The vectors are stored in a vector index that supports efficient
nearest-neighbor search, enabling fast retrieval even across very large document collections.
4. Query Handling and Retrieval
When a user submits a query in NotebookLM, the system converts the query into an embedding using the
same model that was used to embed the document chunks. This ensures that the query and the document

--- Document 2 ---
When a user uploads a document to NotebookLM, the system begins an automatic indexing process. The
document is first parsed to extract its raw text content. For PDFs, this involves optical character recognition
(OCR) if the document contains scanned pages, or direct text extraction for digital PDFs. The extracted text is
then cleaned and normalized to remove formatting artifacts.
Next, the text is split into overlapping chunks using a strategy that preserves semantic coherence. Rather

--- Docum

### Step 6 - Augmentation

The retrieved chunks are concatenated into a single context string.
This context will be injected into the generation prompt to ground the LLM's answer.

This cell aggregates the content from all documents stored in `fused_docs` into a single string variable, `context`. This is crucial because most LLM prompts require a single, cohesive context block rather than a list of separate document objects.


In [12]:
# Join all retrieved chunks into one context block
context = "\n\n".join([doc.page_content for doc in fused_docs])

print(context)

rather than simple keyword matching. The vectors are stored in a vector index that supports efficient
nearest-neighbor search, enabling fast retrieval even across very large document collections.
4. Query Handling and Retrieval
When a user submits a query in NotebookLM, the system converts the query into an embedding using the
same model that was used to embed the document chunks. This ensures that the query and the document

When a user uploads a document to NotebookLM, the system begins an automatic indexing process. The
document is first parsed to extract its raw text content. For PDFs, this involves optical character recognition
(OCR) if the document contains scanned pages, or direct text extraction for digital PDFs. The extracted text is
then cleaned and normalized to remove formatting artifacts.
Next, the text is split into overlapping chunks using a strategy that preserves semantic coherence. Rather

than splitting at fixed character counts, NotebookLM uses intelligent chunking 

### Step 7 - Generation

The context and original query are passed to the LLM via a structured prompt.
The LLM is instructed to answer **only** from the provided context and to say `"I don't know"` if the answer isn't there.

This cell is typically used to define or pass the user's input query into the subsequent RAG pipeline steps. It serves as the primary variable that drives the entire retrieval and generation process.


In [13]:
query


'How does NotebookLM retrieve relevant information from uploaded documents?'

### Contextual Generation Chain

This cell defines and executes a core generation chain. It uses `ChatPromptTemplate` to structure the prompt, ensuring the LLM is strictly constrained to use only the provided context (`{context}`) when answering the user's question (`{question}`). The resulting chain pipes the structured prompt into the Language Model (`llm`) to generate the final answer.


In [12]:
prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant. Use ONLY the context provided below to answer the question.
Be clear, concise, and accurate in your response.
If the answer is not present in the context, say "I don't know" - do not make up an answer.

Context:
{context}

Question: {question}

Answer:
""")

# Chain: prompt -> LLM
generation_chain = prompt | llm

response = generation_chain.invoke({"context": context, "question": query})

print(response.content)

- When a document is uploaded NotebookLM parses the file (using OCR for scanned PDFs or direct extraction for digital PDFs), then cleans and normalizes the extracted text.
- The text is split into overlapping chunks that preserve sentence and paragraph boundaries so semantic coherence is maintained.
- Each chunk is converted to a high-dimensional embedding using an embedding model; those vectors are stored in a vector index that supports efficient nearest-neighbor search.
- When you submit a query, the query is embedded with the same model so it occupies the same semantic space; cosine similarity (nearest-neighbor search) is used to find the top-k most relevant chunks.
- In practice a hybrid approach combining dense (vector) retrieval and sparse (e.g., BM25) matching is likely used, and the model can reference the specific chunks it used to generate an answer.
